# NB2 — data, audit and GPU pilot

Builds the A / B / general development data with the pinned tokenizer and re-derives every answer independently (audit). Then runs the GPU pilot: FP16 vs FP32 check, speed and memory, merge tolerances and the untrained-model (M0) baseline. Writes `pilot/study.yaml`, the settings every later stage uses.

**Accelerator:** GPU T4 x2 · **Internet:** OFF · **Attach:** code, model

Run with **Save Version → Save & Run All (Commit)**. Afterwards: open `pilot/pilot_report.md`. Go on only if it says `precision_validated: True` and lists no concerns. Optionally skim `data/audit/review_sheet.md`.

In [ ]:
SESSION_HOURS = 12   # your GPU session limit (see the notebook settings panel)
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})

In [ ]:
MODEL = kaggle.one(found, 'models', 'pinned model Dataset (made from the NB1 output)')
M0_CONFIG = kaggle.m0_config(MODEL, WORKING / 'configs' / 'm0.yaml')
banks = kaggle.make_banks(WORKING / 'data', tokenizer=MODEL)
assert banks['audit_passed'], 'the independent audit found wrong reference answers; see data/audit/audit.json'
print(banks)
pcl('pilot', '--model-config', M0_CONFIG, '--data-root', WORKING / 'data', '--out', WORKING / 'pilot')
show(WORKING / 'pilot' / 'pilot_report.md')